In [1]:
import torch

print("="*60)

if torch.cuda.is_available():
    print("✅ GPU Available")
    print("GPU :", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
    print("PyTorch:", torch.__version__)
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB")
else:
    print("❌ GPU NOT FOUND")

print("="*60)

✅ GPU Available
GPU : Tesla T4
CUDA: 12.8
PyTorch: 2.11.0+cu128
VRAM : 14.56 GB


In [2]:
%%capture

!pip install -q -U \
"unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
transformers \
datasets \
accelerate \
peft \
trl \
bitsandbytes \
sentencepiece \
evaluate \
tensorboard \
sacrebleu

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import unsloth
import transformers
import peft
import datasets
import trl
import evaluate
import torch

print("✅ All libraries imported successfully")
print("Unsloth:", unsloth.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("TRL:", trl.__version__)
print("Torch:", torch.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
✅ All libraries imported successfully
Unsloth: 2026.7.2
Transformers: 5.5.0
PEFT: 0.19.1
Datasets: 4.3.0
TRL: 0.24.0
Torch: 2.11.0+cu128


In [2]:
# ==========================================================
# Imports
# ==========================================================

import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import torch

from google.colab import drive

from datasets import (
    load_dataset,
    Dataset,
    DatasetDict,
)

from unsloth import FastLanguageModel

from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
)

import evaluate

warnings.filterwarnings("ignore")

print("✅ All imports successful")

✅ All imports successful


In [4]:
# ==========================================================
# Mount Google Drive
# ==========================================================

drive.mount("/content/drive")

# ==========================================================
# Project Directory
# ==========================================================

PROJECT_DIR = "/content/drive/MyDrive/LLM_Finetuning_Assignment"

CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
LOG_DIR        = os.path.join(PROJECT_DIR, "logs")
MODEL_DIR      = os.path.join(PROJECT_DIR, "final_model")
EVAL_DIR       = os.path.join(PROJECT_DIR, "evaluation")
PRED_DIR       = os.path.join(PROJECT_DIR, "predictions")
DATASET_DIR    = os.path.join(PROJECT_DIR, "dataset_splits")

# ==========================================================
# Create folders if they don't exist
# ==========================================================

for folder in [
    PROJECT_DIR,
    CHECKPOINT_DIR,
    LOG_DIR,
    MODEL_DIR,
    EVAL_DIR,
    PRED_DIR,
    DATASET_DIR,
]:
    os.makedirs(folder, exist_ok=True)

print("=" * 60)
print("✅ Project folders ready")
print("=" * 60)

print(PROJECT_DIR)
print("│")
print("├── checkpoints")
print("├── logs")
print("├── final_model")
print("├── evaluation")
print("├── predictions")
print("└── dataset_splits")

Mounted at /content/drive
✅ Project folders ready
/content/drive/MyDrive/LLM_Finetuning_Assignment
│
├── checkpoints
├── logs
├── final_model
├── evaluation
├── predictions
└── dataset_splits


In [5]:
# ==========================================================
# Configuration
# ==========================================================

SEED = 42

MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"

TRAIN_SIZE = 5000
VAL_SIZE = 500
TEST_SIZE = 500

MAX_SEQ_LENGTH = 256
MAX_NEW_TOKENS = 100

LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

NUM_EPOCHS = 3
LEARNING_RATE = 1e-4
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
WEIGHT_DECAY = 0.01

LOGGING_STEPS = 25
EVAL_STEPS = 100
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 2

In [6]:
# ==========================================================
# Load Dataset
# ==========================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

dataset = load_dataset("cfilt/iitb-english-hindi")

print(dataset)

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/500k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 1659083
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 520
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2507
    })
})


In [7]:
# ==========================================================
# Create Dataset Splits
# ==========================================================

full_train = dataset["train"].shuffle(seed=SEED)

train_dataset = full_train.select(range(TRAIN_SIZE))

val_dataset = full_train.select(
    range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE)
)

test_dataset = full_train.select(
    range(
        TRAIN_SIZE + VAL_SIZE,
        TRAIN_SIZE + VAL_SIZE + TEST_SIZE
    )
)

print(f"Train samples      : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"Test samples       : {len(test_dataset)}")

Train samples      : 5000
Validation samples : 500
Test samples       : 500


In [8]:
# ==========================================================
# Save Dataset Splits
# ==========================================================

train_dataset.to_json(
    os.path.join(DATASET_DIR, "train_split.jsonl")
)

val_dataset.to_json(
    os.path.join(DATASET_DIR, "validation_split.jsonl")
)

test_dataset.to_json(
    os.path.join(DATASET_DIR, "test_split.jsonl")
)

print("✅ Dataset splits saved successfully.")
print(DATASET_DIR)

Creating json from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

✅ Dataset splits saved successfully.
/content/drive/MyDrive/LLM_Finetuning_Assignment/dataset_splits


In [10]:
# ==========================================================
# Build Translation Prompt
# ==========================================================

def build_prompt(english_sentence):
    return f"""Translate the following English sentence into Hindi.

### English:
{english_sentence}

### Hindi:
"""

In [11]:
# ==========================================================
# Convert Dataset
# ==========================================================

def convert_dataset(ds):

    rows = []

    for sample in ds:

        english = sample["translation"]["en"].strip()
        hindi = sample["translation"]["hi"].strip()

        rows.append({

            "source": english,

            "target": hindi,

            "prompt": build_prompt(english),

        })

    return pd.DataFrame(rows)


train_df = convert_dataset(train_dataset)
val_df = convert_dataset(val_dataset)
test_df = convert_dataset(test_dataset)

print(train_df.head(3))

                                              source  \
0                                  on the intuition.   
1                                    Ceiba pentandra   
2  Have you not regarded how your Lord dealt with...   

                                              target  \
0                                     अंतर्ज्ञान पर।   
1                                           कण्टकारी   
2  क्या तुमने देखा नहीं कि तुम्हारे आद के साथ क्य...   

                                              prompt  
0  Translate the following English sentence into ...  
1  Translate the following English sentence into ...  
2  Translate the following English sentence into ...  


In [12]:
# ==========================================================
# Build SFT Dataset
# ==========================================================

from datasets import Dataset

def create_training_text(row):
    return row["prompt"] + row["target"]


train_sft_df = train_df.copy()
val_sft_df = val_df.copy()

train_sft_df["text"] = train_sft_df.apply(create_training_text, axis=1)
val_sft_df["text"] = val_sft_df.apply(create_training_text, axis=1)

train_sft_dataset = Dataset.from_pandas(
    train_sft_df[["text"]],
    preserve_index=False,
)

val_sft_dataset = Dataset.from_pandas(
    val_sft_df[["text"]],
    preserve_index=False,
)

print(train_sft_dataset)
print()
print(train_sft_dataset[0]["text"])

Dataset({
    features: ['text'],
    num_rows: 5000
})

Translate the following English sentence into Hindi.

### English:
on the intuition.

### Hindi:
अंतर्ज्ञान पर।


In [13]:
# ==========================================================
# Load Qwen Model
# ==========================================================

model, tokenizer = FastLanguageModel.from_pretrained(

    model_name=MODEL_NAME,

    max_seq_length=MAX_SEQ_LENGTH,

    dtype=None,

    load_in_4bit=True,

)

print("✅ Model loaded successfully")

==((====))==  Unsloth 2026.7.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

✅ Model loaded successfully


In [14]:
# ==========================================================
# Apply LoRA
# ==========================================================

model = FastLanguageModel.get_peft_model(

    model,

    r=LORA_R,

    target_modules=[

        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",

        "gate_proj",
        "up_proj",
        "down_proj",

    ],

    lora_alpha=LORA_ALPHA,

    lora_dropout=LORA_DROPOUT,

    bias="none",

    use_gradient_checkpointing="unsloth",

    random_state=SEED,

)

print("✅ LoRA attached successfully")

Unsloth 2026.7.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


✅ LoRA attached successfully


In [15]:
# ==========================================================
# Tokenize Dataset
# ==========================================================

MAX_SEQ_LENGTH = 256

def tokenize_function(example):

    tokenized = tokenizer(

        example["text"],

        truncation=True,

        max_length=MAX_SEQ_LENGTH,

        padding="max_length",

    )

    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized


train_tokenized = train_sft_dataset.map(

    tokenize_function,

    remove_columns=train_sft_dataset.column_names,

)

val_tokenized = val_sft_dataset.map(

    tokenize_function,

    remove_columns=val_sft_dataset.column_names,

)

print(train_tokenized)
print(val_tokenized)

print("\nSample Keys:")

print(train_tokenized[0].keys())

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 5000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 500
})

Sample Keys:
dict_keys(['input_ids', 'attention_mask', 'labels'])


In [16]:
# ==========================================================
# Data Collator
# ==========================================================

data_collator = DataCollatorForLanguageModeling(

    tokenizer=tokenizer,

    mlm=False,

)

print("✅ Data Collator Ready")

✅ Data Collator Ready


In [20]:
# ==========================================================
# Training Arguments
# ==========================================================

from transformers import TrainingArguments, Trainer

RUN_NAME = "qwen3_lora_run"

CHECKPOINT_DIR = os.path.join(CHECKPOINT_DIR, RUN_NAME)
LOG_DIR = os.path.join(LOG_DIR, RUN_NAME)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

training_args = TrainingArguments(

    output_dir=CHECKPOINT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=1e-4,
    weight_decay=0.01,

    warmup_steps=56,
    lr_scheduler_type="cosine",

    logging_strategy="steps",
    logging_steps=25,

    eval_strategy="steps",
    eval_steps=100,

    save_strategy="steps",
    save_steps=100,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=True,

    report_to="tensorboard",

    remove_unused_columns=False,

    seed=SEED,
)

In [22]:
# ==========================================================
# Start Fine-tuning
# ==========================================================

print("=" * 70)
print("🚀 Starting Fine-tuning...")
print("=" * 70)

train_result = Trainer.train()

print("\n✅ Fine-tuning Complete!")
print(train_result)

🚀 Starting Fine-tuning...


TypeError: Trainer.train() missing 1 required positional argument: 'self'